# Road Surface Classification — Vibration Embedding & Clustering
> **Pipeline:** Denoised Z-score normalised accelerometer (X,Y,Z)  
> **Goal:** Cluster windows into distinct road-surface types  
> **Embedding strategies compared:** Hand-crafted features  vs  ROCKET kernels

---
### Table of Contents
1. [Install & Imports](#1)
2. [Configuration](#2)
3. [Data Loading & Windowing](#3)
4. [Embedding Strategy A — Hand-Crafted Features](#4)
5. [Embedding Strategy B — ROCKET](#5)
6. [Post-Processing (PCA + L2 Norm)](#6)
7. [Find Optimal Number of Clusters](#7)
8. [Clustering](#8)
9. [Evaluation & Comparison](#9)
10. [Visualization](#10)
11. [Cluster Interpretation](#11)
12. [Save Results](#12)


## 1. Install & Imports <a id='1'></a>

In [1]:
# Install required packages (run once)
# !pip install numpy scipy scikit-learn matplotlib pandas umap-learn aeon

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import kurtosis, skew
from scipy.signal import welch

from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score,
    normalized_mutual_info_score
)

# Optional — UMAP for visualization
try:
    from umap import UMAP
    UMAP_AVAILABLE = True
    print('✅ UMAP available')
except ImportError:
    from sklearn.manifold import TSNE
    UMAP_AVAILABLE = False
    print('⚠️  UMAP not found — will use t-SNE. Install: pip install umap-learn')

# Optional — ROCKET
try:
    from aeon.transformations.collection.convolution_based import Rocket
    ROCKET_AVAILABLE = True
    print('✅ ROCKET (aeon) available')
except ImportError:
    ROCKET_AVAILABLE = False
    print('⚠️  aeon not found — ROCKET disabled. Install: pip install aeon')

print('\n✅ All imports done')


✅ UMAP available
⚠️  aeon not found — ROCKET disabled. Install: pip install aeon

✅ All imports done


## 2. Configuration <a id='2'></a>

In [ ]:
CONFIG = {
    # ── Data ──────────────────────────────────────────
    'fs':               1000,    # Sampling frequency (Hz)
    'window_size_sec':  3.0,     # Window length in seconds
    'overlap':          0.5,     # 50% overlap between windows

    # ── Frequency bands for road surface features ─────
    # Tune these to your sensor's frequency range
    'freq_bands': [
        (1,   10),   # Low  — vehicle body motion
        (10,  30),   # Mid-low — smooth asphalt
        (30,  60),   # Mid  — rough asphalt
        (60,  100),  # Mid-high — gravel / cobblestone
        (100, 150),  # High — fine gravel / texture
    ],

    # ── Clustering ────────────────────────────────────
    'n_clusters':         None,         # None = auto-detect, or set int e.g. 4
    'k_range':            (2, 9),        # Search range when n_clusters=None
    'clustering_method':  'kmeans',      # 'kmeans' | 'agglomerative' | 'dbscan'

    # DBSCAN-specific (only used if clustering_method='dbscan')
    'dbscan_eps':         0.25,
    'dbscan_min_samples': 10,

    # ── Embedding ─────────────────────────────────────
    'rocket_kernels':     10_000,        # Number of ROCKET kernels
    'pca_variance':       0.95,          # Variance to retain after PCA
}

print('Configuration loaded ✅')
for k, v in CONFIG.items():
    print(f'  {k:25s}: {v}')


## 3. Data Loading & Windowing <a id='3'></a>

**Replace the synthetic data block with your real CSV loading.**
Expected CSV format:
```
acc_x, acc_y, acc_z          ← required (already denoised & z-scored)
label                        ← optional ground truth surface name
timestamp                    ← optional
```


In [ ]:
# ── OPTION A: Load from CSV (your real data) ──────────────────────────────
# Uncomment and edit the path below:

# def load_csv(filepath):
#     df = pd.read_csv(filepath)
#     # Auto-detect column names
#     col_map = {}
#     for col in df.columns:
#         cl = col.lower()
#         if 'x' in cl and 'acc' in cl:              col_map['x'] = col
#         elif 'y' in cl and 'acc' in cl:            col_map['y'] = col
#         elif 'z' in cl and 'acc' in cl:            col_map['z'] = col
#         elif any(k in cl for k in ['label','surface','class']): col_map['label'] = col
#     data      = df[[col_map['x'], col_map['y'], col_map['z']]].values.astype(np.float32)
#     gt_labels = df[col_map['label']].values if 'label' in col_map else None
#     return data, gt_labels
#
# data, GT_LABELS = load_csv('your_data.csv')


# ── OPTION B: Synthetic demo data ─────────────────────────────────────────
np.random.seed(42)
fs  = CONFIG['fs']
dur = 30   # seconds per surface

surface_profiles = {
    #  name          noise   band weights (matching freq_bands above)
    'asphalt':      (0.05, [0.60, 0.30, 0.05, 0.03, 0.02]),
    'gravel':       (0.30, [0.10, 0.20, 0.35, 0.25, 0.10]),
    'cobblestone':  (0.20, [0.20, 0.50, 0.20, 0.06, 0.04]),
    'dirt':         (0.25, [0.15, 0.20, 0.30, 0.20, 0.15]),
}

segments, labels_list = [], []
band_centers = [5, 20, 45, 80, 125]   # representative Hz per band

for surface, (noise_std, band_weights) in surface_profiles.items():
    n = fs * dur
    t = np.linspace(0, dur, n)
    sig = np.zeros((n, 3))
    for amp, freq in zip(band_weights, band_centers):
        for ax in range(3):
            phase = np.random.uniform(0, 2 * np.pi)
            sig[:, ax] += amp * np.sin(2 * np.pi * freq * t + phase)
    sig += np.random.randn(n, 3) * noise_std
    # Z-score (already done in your data)
    sig = (sig - sig.mean(0)) / (sig.std(0) + 1e-8)
    segments.append(sig)
    labels_list.extend([surface] * n)

DATA      = np.vstack(segments).astype(np.float32)
GT_LABELS = np.array(labels_list)

print(f'Data shape : {DATA.shape}  (samples × axes)')
print(f'Duration   : {len(DATA)/fs:.1f} s  |  Fs = {fs} Hz')
print(f'Surfaces   : {np.unique(GT_LABELS)}')

# Quick raw signal preview
fig, axes = plt.subplots(len(surface_profiles), 3, figsize=(15, 8), sharex=True)
fig.suptitle('Raw Accelerometer Signal per Surface (first 2 s)', fontsize=13, fontweight='bold')
t_preview = np.arange(2 * fs) / fs
for row, (surface, _) in enumerate(surface_profiles.items()):
    start = row * dur * fs
    for col, ax_name in enumerate(['X', 'Y', 'Z']):
        axes[row, col].plot(t_preview, DATA[start:start + 2*fs, col], lw=0.7)
        if row == 0: axes[row, col].set_title(f'Axis {ax_name}', fontweight='bold')
        if col == 0: axes[row, col].set_ylabel(surface, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Windowing ─────────────────────────────────────────────────────────────
def create_windows(data, fs, window_sec, overlap, gt_labels=None):
    win_len = int(fs * window_sec)
    step    = int(win_len * (1 - overlap))
    windows, win_labels = [], []
    for start in range(0, len(data) - win_len + 1, step):
        windows.append(data[start:start + win_len])     # (win_len, 3)
        if gt_labels is not None:
            seg_lbl = gt_labels[start:start + win_len]
            unique, counts = np.unique(seg_lbl, return_counts=True)
            win_labels.append(unique[np.argmax(counts)])  # majority label
    print(f'Windows created : {len(windows)}')
    print(f'Window length   : {win_len} samples  ({window_sec} s)')
    print(f'Step size       : {step} samples  ({overlap*100:.0f}% overlap)')
    return windows, (win_labels if gt_labels is not None else None)

WINDOWS, WINDOW_GT = create_windows(
    DATA, CONFIG['fs'], CONFIG['window_size_sec'],
    CONFIG['overlap'], GT_LABELS
)

print(f'\nWindow array shape: {np.array(WINDOWS).shape}')
if WINDOW_GT:
    from collections import Counter
    print('Windows per surface:', dict(Counter(WINDOW_GT)))


## 4. Embedding Strategy A — Hand-Crafted Features <a id='4'></a>

Extracts **~65 physics-informed features** per window across three groups:

| Group | Features | Why it matters |
|-------|----------|----------------|
| Time domain (per axis) | RMS, Peak, Crest Factor, Kurtosis, Skewness, Roughness | Kurtosis → gravel/cobble impulsiveness |
| Frequency domain (per axis) | Band energy **ratios**, Spectral centroid, Dominant freq | Ratios are speed-invariant |
| Cross-axis | X-Z, Y-Z, X-Y correlations, Axis dominance ratios | Cobblestone creates strong Z-X coupling |


In [ ]:
def extract_handcrafted(window, fs, freq_bands):
    """
    window : np.array (win_len, 3) — already denoised & z-scored
    Returns: 1D feature vector
    """
    features = []

    # ── A. Per-axis features ────────────────────────────────────────────────
    for axis in window.T:   # iterate X, Y, Z
        rms  = np.sqrt(np.mean(axis**2)) + 1e-8
        peak = np.max(np.abs(axis))

        # Time domain
        features += [
            rms,
            peak,
            peak / rms,                                  # crest factor
            kurtosis(axis),                              # impulsiveness
            skew(axis),
            np.percentile(np.abs(axis), 95),
            np.percentile(np.abs(axis), 75),
            np.mean(np.abs(np.diff(axis))),              # roughness
            np.std(np.diff(axis)),                       # roughness variability
            np.sum(axis**2),                             # total energy
        ]

        # Frequency domain — band energy RATIOS (speed-invariant)
        freqs, psd = welch(axis, fs=fs, nperseg=min(512, len(axis) // 2))
        total_pwr  = np.sum(psd) + 1e-8
        for f_lo, f_hi in freq_bands:
            mask = (freqs >= f_lo) & (freqs < f_hi)
            features.append(np.sum(psd[mask]) / total_pwr)   # ratio ✅

        # Spectral shape
        centroid = np.sum(freqs * psd) / total_pwr
        spread   = np.sqrt(np.sum((freqs - centroid)**2 * psd) / total_pwr)
        features += [
            freqs[np.argmax(psd)],   # dominant frequency
            centroid,
            spread,
        ]

    # ── B. Cross-axis features ──────────────────────────────────────────────
    x, y, z = window[:, 0], window[:, 1], window[:, 2]
    features += [
        np.corrcoef(x, z)[0, 1],                      # X-Z coupling
        np.corrcoef(y, z)[0, 1],                      # Y-Z coupling
        np.corrcoef(x, y)[0, 1],                      # X-Y coupling
        np.std(z) / (np.std(x) + 1e-8),              # vertical vs lateral
        np.std(z) / (np.std(y) + 1e-8),              # vertical vs longitudinal
    ]

    # ── C. Magnitude channel ────────────────────────────────────────────────
    mag     = np.sqrt(x**2 + y**2 + z**2)
    rms_mag = np.sqrt(np.mean(mag**2)) + 1e-8
    features += [
        np.mean(mag),
        np.std(mag),
        np.max(mag) / rms_mag,   # magnitude crest factor
        kurtosis(mag),
        np.mean(np.abs(np.diff(mag))),
    ]

    return np.array(features, dtype=np.float32)


# Generate embeddings for all windows
EMB_HC = np.array([
    extract_handcrafted(w, CONFIG['fs'], CONFIG['freq_bands'])
    for w in WINDOWS
])
EMB_HC = np.nan_to_num(EMB_HC, nan=0.0, posinf=0.0, neginf=0.0)

print(f'Hand-crafted embedding shape: {EMB_HC.shape}')
print(f'  → {EMB_HC.shape[0]} windows  ×  {EMB_HC.shape[1]} features')


## 5. Embedding Strategy B — ROCKET <a id='5'></a>

> **ROCKET** = RandOm Convolutional KErnel Transform  
> Applies 10,000 random convolutional kernels and extracts **PPV** (proportion of positive values) and **max** per kernel → 20,000-dim embedding.  
> No training required. State-of-the-art on time series benchmarks.

```
Window (3000×3)  →  ROCKET (10k random kernels)  →  20,000-dim vector
                                                         ↓
                                                    PCA → 30–80 dims
```


In [ ]:
if ROCKET_AVAILABLE:
    # ROCKET expects shape (n_windows, n_axes, window_length)
    X_rocket = np.array([w.T for w in WINDOWS])   # (N, 3, win_len)

    print(f'Input to ROCKET: {X_rocket.shape}')
    print('Fitting & transforming ROCKET kernels (may take ~30s)...')

    rocket = Rocket(num_kernels=CONFIG['rocket_kernels'], random_state=42)
    rocket.fit(X_rocket)
    EMB_ROCKET = rocket.transform(X_rocket).astype(np.float32)
    EMB_ROCKET = np.nan_to_num(EMB_ROCKET, nan=0.0, posinf=0.0, neginf=0.0)

    print(f'ROCKET embedding shape: {EMB_ROCKET.shape}')
    print(f'  → {EMB_ROCKET.shape[0]} windows  ×  {EMB_ROCKET.shape[1]} features')
else:
    EMB_ROCKET = None
    print('⚠️  ROCKET not available — skipping. Install with: pip install aeon')
    print('   Hand-crafted embeddings will be used for all downstream steps.')


## 6. Post-Processing — L2 Normalise + PCA <a id='6'></a>

| Step | Why |
|------|-----|
| **L2 Normalise** | Enables cosine distance — direction matters, not magnitude |
| **PCA** | Removes noisy dimensions, speeds up clustering, reduces curse of dimensionality |


In [ ]:
def postprocess(emb, label, variance=0.95):
    """L2 normalise then PCA."""
    emb_norm = normalize(emb, norm='l2')
    pca      = PCA(n_components=variance, random_state=42)
    emb_pca  = pca.fit_transform(emb_norm)
    print(f'[{label}] {emb.shape[1]:>6} dims  →  L2 norm  →  PCA {emb_pca.shape[1]} dims '
          f'({variance*100:.0f}% variance)')
    return emb_norm, emb_pca, pca

print('── Hand-Crafted ──')
HC_NORM, HC_PCA, HC_PCA_MODEL = postprocess(EMB_HC, 'Hand-Crafted', CONFIG['pca_variance'])

if EMB_ROCKET is not None:
    print('\n── ROCKET ──')
    RK_NORM, RK_PCA, RK_PCA_MODEL = postprocess(EMB_ROCKET, 'ROCKET', CONFIG['pca_variance'])
else:
    RK_NORM = RK_PCA = None


## 7. Find Optimal Number of Clusters <a id='7'></a>

In [ ]:
def find_optimal_k(emb_pca, k_range, label=''):
    ks          = range(k_range[0], k_range[1] + 1)
    inertias, sil, db, ch = [], [], [], []

    for k in ks:
        km  = KMeans(n_clusters=k, n_init=20, random_state=42, max_iter=500)
        lbl = km.fit_predict(emb_pca)
        inertias.append(km.inertia_)
        sil.append(silhouette_score(emb_pca, lbl))
        db.append(davies_bouldin_score(emb_pca, lbl))
        ch.append(calinski_harabasz_score(emb_pca, lbl))

    best_k = list(ks)[np.argmax(sil)]

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'Optimal K — {label}', fontsize=13, fontweight='bold')
    titles  = ['Elbow (Inertia) ↓', 'Silhouette ↑', 'Davies-Bouldin ↓']
    metrics = [inertias, sil, db]
    colors  = ['royalblue', 'seagreen', 'tomato']
    for ax, title, vals, c in zip(axes, titles, metrics, colors):
        ax.plot(ks, vals, f'o-', color=c, lw=2, markersize=7)
        ax.axvline(best_k, color='black', linestyle='--', alpha=0.6, label=f'Best K={best_k}')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('K')
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f'\n  K search results ({label}):')
    print(f'  {"K":>4}  {"Silhouette":>12}  {"Davies-Bouldin":>16}  {"CH Score":>10}')
    for k, s, d, c in zip(ks, sil, db, ch):
        marker = ' ◀ best' if k == best_k else ''
        print(f'  {k:>4}  {s:>12.4f}  {d:>16.4f}  {c:>10.1f}{marker}')
    print(f'\n  Recommended K = {best_k}')
    return best_k, sil


N_CLUSTERS = CONFIG.get('n_clusters')

if N_CLUSTERS is None:
    print('━' * 55)
    print('Hand-Crafted Embeddings')
    print('━' * 55)
    N_CLUSTERS_HC, SIL_HC = find_optimal_k(HC_PCA, CONFIG['k_range'], 'Hand-Crafted')

    if RK_PCA is not None:
        print('\n' + '━' * 55)
        print('ROCKET Embeddings')
        print('━' * 55)
        N_CLUSTERS_RK, SIL_RK = find_optimal_k(RK_PCA, CONFIG['k_range'], 'ROCKET')
    else:
        N_CLUSTERS_RK = N_CLUSTERS_HC
else:
    N_CLUSTERS_HC = N_CLUSTERS_RK = N_CLUSTERS
    print(f'Using user-specified K = {N_CLUSTERS}')


## 8. Clustering <a id='8'></a>

| Method | When to use |
|--------|-------------|
| `kmeans` | You know K, fast, good general baseline |
| `agglomerative` | Best cluster separation, cosine metric |
| `dbscan` | Unknown K, automatically finds transition zones |


In [ ]:
def cluster(emb_norm, emb_pca, n_clusters, method, cfg):
    if method == 'kmeans':
        model  = KMeans(n_clusters=n_clusters, n_init=30, max_iter=500, random_state=42)
        labels = model.fit_predict(emb_pca)

    elif method == 'agglomerative':
        model  = AgglomerativeClustering(n_clusters=n_clusters, metric='cosine', linkage='average')
        labels = model.fit_predict(emb_norm)

    elif method == 'dbscan':
        model  = DBSCAN(eps=cfg['dbscan_eps'], min_samples=cfg['dbscan_min_samples'], metric='cosine')
        labels = model.fit_predict(emb_norm)
        n_found = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = np.sum(labels == -1)
        print(f'  DBSCAN → {n_found} clusters | {n_noise} noise points ({n_noise/len(labels)*100:.1f}%)')

    # Distribution summary
    unique, counts = np.unique(labels, return_counts=True)
    print(f'  Cluster distribution:')
    for u, c in zip(unique, counts):
        tag = ' (noise/transitions)' if u == -1 else ''
        bar = '█' * int(c / len(labels) * 40)
        print(f'    Cluster {u:>2}{tag}: {c:>4} windows ({c/len(labels)*100:5.1f}%)  {bar}')
    return labels


METHOD = CONFIG['clustering_method']

print(f'Clustering method: {METHOD.upper()}\n')
print('── Hand-Crafted ──')
LABELS_HC = cluster(HC_NORM, HC_PCA, N_CLUSTERS_HC, METHOD, CONFIG)

if RK_NORM is not None:
    print('\n── ROCKET ──')
    LABELS_RK = cluster(RK_NORM, RK_PCA, N_CLUSTERS_RK, METHOD, CONFIG)
else:
    LABELS_RK = None


## 9. Evaluation & Comparison <a id='9'></a>

In [ ]:
def evaluate(emb_pca, labels, gt_labels=None, tag=''):
    valid = labels != -1
    lv, ev = labels[valid], emb_pca[valid]
    if len(np.unique(lv)) < 2:
        print(f'[{tag}] Only 1 cluster — metrics unavailable'); return {}

    m = {
        'Silhouette':        silhouette_score(ev, lv),
        'Davies-Bouldin':    davies_bouldin_score(ev, lv),
        'Calinski-Harabasz': calinski_harabasz_score(ev, lv),
    }
    if gt_labels is not None:
        gv = np.array(gt_labels)[valid]
        m['ARI'] = adjusted_rand_score(gv, lv)
        m['NMI'] = normalized_mutual_info_score(gv, lv)

    # Quality label
    s = m['Silhouette']
    q = 'Excellent' if s > 0.6 else 'Good' if s > 0.4 else 'Fair' if s > 0.2 else 'Poor'
    m['Quality'] = q
    return m


METRICS_HC = evaluate(HC_PCA, LABELS_HC, WINDOW_GT, 'Hand-Crafted')
METRICS_RK = evaluate(RK_PCA, LABELS_RK, WINDOW_GT, 'ROCKET') if LABELS_RK is not None else None

# ── Comparison table ────────────────────────────────────────────────────────
rows = {'Hand-Crafted': METRICS_HC}
if METRICS_RK: rows['ROCKET'] = METRICS_RK
comp_df = pd.DataFrame(rows).T

print('\n' + '='*60)
print('  EMBEDDING COMPARISON')
print('='*60)
print(comp_df.to_string())
print()
print('Interpretation:')
print('  Silhouette  → higher is better  (range -1 to 1)')
print('  Davies-Bouldin → lower is better')
print('  ARI / NMI   → closer to 1.0 = matches ground truth')

if METRICS_RK:
    winner = 'ROCKET' if METRICS_RK['Silhouette'] > METRICS_HC['Silhouette'] else 'Hand-Crafted'
    print(f'\n  🏆 Better embedding for your data: {winner}')


## 10. Visualization <a id='10'></a>

In [ ]:
def reduce_2d(emb_norm, metric='cosine'):
    if UMAP_AVAILABLE:
        r = UMAP(n_components=2, metric=metric, n_neighbors=30, min_dist=0.1, random_state=42)
        return r.fit_transform(emb_norm), 'UMAP'
    else:
        r = TSNE(n_components=2, metric=metric, perplexity=30, random_state=42, n_iter=1000)
        return r.fit_transform(emb_norm), 't-SNE'


def plot_clusters(emb_2d, labels, gt_labels, title, method_name):
    n_plots = 2 if gt_labels is not None else 1
    fig, axes = plt.subplots(1, n_plots, figsize=(8 * n_plots, 6))
    if n_plots == 1: axes = [axes]

    palette = plt.cm.tab10(np.linspace(0, 1, max(len(set(labels)), 2)))

    # Predicted clusters
    ax = axes[0]
    for i, lbl in enumerate(sorted(set(labels))):
        mask = labels == lbl
        if lbl == -1:
            ax.scatter(emb_2d[mask,0], emb_2d[mask,1], c='lightgray', s=8, alpha=0.3, label='Transition')
        else:
            ax.scatter(emb_2d[mask,0], emb_2d[mask,1], color=palette[i], s=12, alpha=0.7, label=f'Cluster {lbl}')
    ax.set_title(f'{title}\nPredicted Clusters ({method_name})', fontweight='bold')
    ax.set_xlabel(f'{method_name}-1'); ax.set_ylabel(f'{method_name}-2')
    ax.legend(markerscale=2, loc='best'); ax.grid(alpha=0.2)

    # Ground truth
    if gt_labels is not None:
        ax2 = axes[1]
        gt  = np.array(gt_labels)
        unique_gt = np.unique(gt)
        gt_palette = plt.cm.Set2(np.linspace(0, 1, len(unique_gt)))
        for i, surf in enumerate(unique_gt):
            mask = gt == surf
            ax2.scatter(emb_2d[mask,0], emb_2d[mask,1], color=gt_palette[i], s=12, alpha=0.7, label=surf)
        ax2.set_title(f'{title}\nGround Truth Labels ({method_name})', fontweight='bold')
        ax2.set_xlabel(f'{method_name}-1'); ax2.set_ylabel(f'{method_name}-2')
        ax2.legend(markerscale=2, loc='best'); ax2.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()


print('Reducing Hand-Crafted embeddings to 2D...')
HC_2D, DIM_METHOD = reduce_2d(HC_NORM)
plot_clusters(HC_2D, LABELS_HC, WINDOW_GT, 'Hand-Crafted', DIM_METHOD)

if RK_NORM is not None:
    print('Reducing ROCKET embeddings to 2D...')
    RK_2D, _ = reduce_2d(RK_NORM)
    plot_clusters(RK_2D, LABELS_RK, WINDOW_GT, 'ROCKET', DIM_METHOD)


## 11. Cluster Interpretation <a id='11'></a>

What makes each cluster unique? We look at the **top discriminative features** to understand what each cluster represents physically.


In [ ]:
def interpret_clusters(embeddings, labels, top_n=10):
    df = pd.DataFrame(embeddings)
    df['cluster'] = labels
    profile = df.groupby('cluster').mean()

    # Most discriminative features = highest variance across cluster means
    feature_var = np.var(profile.values, axis=0)
    top_feats   = np.argsort(feature_var)[::-1][:top_n]

    # Heatmap
    subset = profile.iloc[:, top_feats]
    fig, ax = plt.subplots(figsize=(13, max(3, len(profile) * 0.9)))
    im = ax.imshow(subset.values, aspect='auto', cmap='RdYlGn')
    ax.set_xticks(range(top_n))
    ax.set_xticklabels([f'Feat {i}' for i in top_feats], rotation=45, ha='right')
    ax.set_yticks(range(len(profile)))
    ax.set_yticklabels(['Noise' if i == -1 else f'Surface {i}' for i in profile.index])
    plt.colorbar(im, ax=ax, label='Mean Value')
    ax.set_title('Cluster Feature Profiles — Top Discriminative Features', fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()

    # Radar / bar chart per cluster
    n_clusters_plot = len([l for l in profile.index if l != -1])
    fig, axes = plt.subplots(1, n_clusters_plot, figsize=(5 * n_clusters_plot, 4), sharey=True)
    if n_clusters_plot == 1: axes = [axes]
    palette = plt.cm.tab10(np.linspace(0, 1, n_clusters_plot))
    plot_idx = 0
    for cluster_id in profile.index:
        if cluster_id == -1: continue
        vals = subset.loc[cluster_id].values
        axes[plot_idx].barh(range(top_n), vals, color=palette[plot_idx], alpha=0.8)
        axes[plot_idx].set_yticks(range(top_n))
        axes[plot_idx].set_yticklabels([f'F{i}' for i in top_feats])
        axes[plot_idx].set_title(f'Surface {cluster_id}', fontweight='bold')
        axes[plot_idx].grid(axis='x', alpha=0.3)
        plot_idx += 1
    plt.suptitle('Feature Profiles per Cluster', fontweight='bold')
    plt.tight_layout()
    plt.show()

    return profile, top_feats


print('── Hand-Crafted Cluster Profiles ──')
HC_PROFILE, HC_TOP_FEATS = interpret_clusters(EMB_HC, LABELS_HC)

if LABELS_RK is not None:
    print('\n── ROCKET Cluster Profiles ──')
    RK_PROFILE, RK_TOP_FEATS = interpret_clusters(EMB_ROCKET, LABELS_RK)


## 12. Save Results <a id='12'></a>

In [ ]:
# ── Save Hand-Crafted results ───────────────────────────────────────────────
hc_df = pd.DataFrame(EMB_HC, columns=[f'feat_{i}' for i in range(EMB_HC.shape[1])])
hc_df['cluster']  = LABELS_HC
hc_df['umap_x']   = HC_2D[:, 0]
hc_df['umap_y']   = HC_2D[:, 1]
if WINDOW_GT: hc_df['gt_label'] = WINDOW_GT
hc_df.to_csv('results_handcrafted.csv', index=False)
print('Saved: results_handcrafted.csv')

# ── Save ROCKET results ─────────────────────────────────────────────────────
if LABELS_RK is not None:
    rk_df = pd.DataFrame()
    rk_df['cluster']  = LABELS_RK
    rk_df['umap_x']   = RK_2D[:, 0]
    rk_df['umap_y']   = RK_2D[:, 1]
    if WINDOW_GT: rk_df['gt_label'] = WINDOW_GT
    rk_df.to_csv('results_rocket.csv', index=False)
    print('Saved: results_rocket.csv')

# ── Save metrics ────────────────────────────────────────────────────────────
rows = {'hand_crafted': METRICS_HC}
if METRICS_RK: rows['rocket'] = METRICS_RK
pd.DataFrame(rows).T.to_csv('metrics_comparison.csv')
print('Saved: metrics_comparison.csv')

print('\n✅ All results saved.')
print(comp_df.to_string())
